# ️ Glu-Stock: 02_STRATEGIC_BRAIN
**Phase**: Strategic Intelligence (The Auditor) | v18.26.4 (Transformers-Hub)

This notebook uses the **Qwen2.5-Coder-7B** model via the `transformers` pipeline for high-hierarchy strategic signal auditing.
**v18.26.4 Update**: Transitioned to Qwen2.5-Coder for superior logic and JSON reliability.

In [ ]:
# [INSTALL] Optimized for Transformers + GPU
!pip install -q transformers accelerate bitsandbytes torch yfinance firebase-admin ta

In [ ]:
import json, os, firebase_admin, yfinance as yf, pandas as pd, time, re, torch
from transformers import pipeline
from firebase_admin import credentials, firestore
from datetime import datetime
from kaggle_secrets import UserSecretsClient

class StrategicBrain:
    def __init__(self, model_id="unsloth/Qwen2.5-Coder-7B-Instruct"):
        print(f"Initializing Strategic Brain ({model_id})...", flush=True)
        self.pipe = pipeline(
            "text-generation",
            model=model_id,
            model_kwargs={"torch_dtype": torch.bfloat16},
            device_map="auto",
        )
        print("Model loaded successfully!", flush=True)
        
    def audit_signal(self, ticker, context_data):
        prompt = f"""
        Role: Senior Quantitative Strategist.
        Task: Audit a BUY signal for {ticker} based on 14-day technical history.
        
        CONTEXT (OHLCV Data):
        {context_data}
        
        Instruction: Reply ONLY with valid JSON. Decision must be PROCEED or VETO.
        Format:
        {{
            "decision": "...",
            "reason": "..."
        }}
        """
        messages = [
            {"role": "system", "content": "You are a strict financial strategic auditor. You only output valid JSON."},
            {"role": "user", "content": prompt},
        ]
        
        try:
            outputs = self.pipe(messages, max_new_tokens=256, do_sample=False, pad_token_id=self.pipe.tokenizer.eos_token_id)
            response = outputs[0]["generated_text"][-1]["content"]
            
            # Extract JSON
            match = re.search(r'\{.*\}', response, re.DOTALL)
            clean_json = match.group(0) if match else response
            return json.loads(clean_json)
        except Exception as e:
            return {"decision": "VETO", "reason": f"Audit Fail: {str(e)}"}

def run_strategic_audit():
    print("--- GLU-STOCK STRATEGIC AUDIT STARTING ---", flush=True)
    secrets = UserSecretsClient()
    cred_json = json.loads(secrets.get_secret("FIREBASE_KEY_JSON"))
    if not firebase_admin._apps: firebase_admin.initialize_app(credentials.Certificate(cred_json))
    db = firestore.client()
    
    pending = db.collection("glu_stock_queue_signals").get()
    if not pending: 
        print("[IDLE] No preliminary signals found to audit.", flush=True)
        return
    
    brain = StrategicBrain()
    for doc in pending:
        payload = doc.to_dict().get('payload', {})
        for ticker, info in payload.items():
            print(f"Auditing {ticker}...", flush=True)
            df = yf.download(ticker, period='30d', progress=False)
            audit = brain.audit_signal(ticker, df.tail(14).to_string())
            
            if audit.get('decision') == 'PROCEED':
                db.collection("glu_stock_final_signals").add({
                    'ticker': ticker, 
                    'audit': audit, 
                    'timestamp': datetime.now().isoformat()
                })
                print(f"[APPROVED] {ticker}: {audit.get('reason')}", flush=True)
            else:
                print(f"[VETOED] {ticker}: {audit.get('reason')}", flush=True)
            doc.reference.delete()

if __name__ == "__main__":
    run_strategic_audit()